# Order duration on the TruLaser Center 7030

This notebook reads `customers_sales_with_dxf_data.xlsx` and calculates the **total duration of every order** on the **TRUMPF TruLaser Center 7030**.

## Calculation model

| Step | Quantity | Calculation |
|:-:|---|---|
| 1 | `ejection_method` | from the part size (bounding box): **SmartGate** (small parts) or **SortMaster Speed** (larger parts) |
| 2 | `cutting_time_s` | `cutting_length / cutting_speed` |
| 3 | `ejection_time_s` | constant per ejection method |
| 4 | `time_per_part_s` | `cutting_time_s + ejection_time_s` |
| 5 | `parts_per_sheet` | how many parts fit on one sheet (integer) |
| 6 | `sheets_needed` | `ceil(order_quantity / parts_per_sheet)` |
| 7 | `sheet_changes` | number of sheet changes |
| 8 | `total_time_s` | `order_quantity * time_per_part_s + sheet_changes * sheet_change_time` |

**Not considered** (intentionally, as specified): the placement of the parts on the sheet is not optimized per part.

**Important:** The cells build on each other and must run **in this order**.

**Note:** Comments and explanations are in English. Console messages, `status` texts, column names of the Excel output and Excel sheet names (`Daten`, `Nicht berechnet`, `Zeit je Kunde`) are still in German, so the code behaves exactly as in the German version.

In [9]:
import os                                                           # <<< NEW (imports; may also be moved to the top)
import re
import numpy as np
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.collections import PatchCollection
from shapely.geometry.polygon import orient
import ezdxf

## 1. Machine data and model assumptions

**What happens:** All constants are defined in **one** place so they are easy to check and change.

**Taken from the published TRUMPF data** (TruLaser Center 7030):

| Constant | Value | Meaning |
|---|---|---|
| Sheet size | 3000 × 1500 mm | working area X / Y |
| SmartGate | 30 × 30 to 160 × 160 mm | ejection of **small** parts |
| SortMaster Speed | 90 × 60 to 2000 × 1500 mm | ejection of **larger** parts |

**Not contained in the public manufacturer data, therefore set to `None` and to be filled in by you:**
- `CUTTING_SPEED_MM_PER_MIN`: cutting speed (mm/min)
- `EJECTION_TIME_S`: ejection time per method (seconds per part)
- `SHEET_CHANGE_TIME_S`: time for one sheet change (seconds)

CUTTING_SPEED_MM_PER_MIN: is not explicitly stated but comparable to similar machines, cutting speed of comparable machines can be viewed on page 32 of the following Trumpf brochure:
"https://www.trumpf.com/filestorage/TRUMPF_SG/user_upload/TRUMPF-2D-laser-cutting-machines-brochure-EN.pdf"

EJECTION_TIME_S: is not explicitly stated, we assumed a reasonable speed of 5 seconds for the ejection with the SMARTGATE and 8 seconds for the retrieval with the SORTMASTER because it is main-time-parallel

SHEET_CHANGE_TIME_S: is not explicitly stated, we assumed a reasonable speed of 30 to retrieve the processed sheet and place a new sheet in the chamber

These values depend on material, sheet thickness and laser power or on your machine configuration and are not in the data sheet. **No values were invented.** As long as they are missing, the calculation in step 5 stops with a clear message. All cells before it already run.

**Model decisions (changeable):**
- `OVERLAP_METHOD`: Parts that fall into both ranges (e.g. 100 × 100 mm) could go through SmartGate **or** SortMaster. Default: `"SmartGate"`.
- `NESTING_METHOD`: `"area"` = as requested, via the bounding box area (`sheet area // part area`). `"grid"` = geometric check of how many rectangles fit onto the sheet in rows (both orientations). See the note in step 4.
- `COUNT_FIRST_SHEET_LOAD`: `True` = every required sheet counts as one sheet change (including the first loading). `False` = only the changes *between* sheets (`sheets_needed - 1`).

**Assumption on units:** DXF lengths and dimensions are interpreted as **millimeters**.

In [1]:
MACHINE = "TruLaser Center 7030"

# ---------- Aus den TRUMPF-Herstellerdaten ----------
SHEET_X_MM = 3000                 # Blechformat X
SHEET_Y_MM = 1500                 # Blechformat Y

SMARTGATE_MIN_MM  = (30, 30)      # kleinste / größte Teilegröße SmartGate
SMARTGATE_MAX_MM  = (160, 160)
SORTMASTER_MIN_MM = (90, 60)      # kleinste / größte Teilegröße SortMaster Speed
SORTMASTER_MAX_MM = (2000, 1500)

# ---------- NICHT in den öffentlichen Daten: bitte eintragen ----------
CUTTING_SPEED_MM_PER_MIN = 140000   # z. B. 12000  -> mm pro Minute
EJECTION_TIME_S = {
    "SmartGate": 5,            # Sekunden pro Teil
    "SortMaster Speed": 8,     # Sekunden pro Teil
}
SHEET_CHANGE_TIME_S = 30        # Sekunden pro Blechwechsel

# ---------- Modellentscheidungen ----------
OVERLAP_METHOD = "SmartGate"      # bei Teilen, die in beide Bereiche passen
NESTING_METHOD = "area"           # "area" (Bounding-Box-Fläche) oder "grid" (Reihen-Packung)
COUNT_FIRST_SHEET_LOAD = True     # True: sheet_changes = sheets_needed, False: sheets_needed - 1


def require_machine_data():
    """Bricht mit klarer Meldung ab, wenn Maschinenwerte fehlen."""
    missing = []
    if CUTTING_SPEED_MM_PER_MIN is None:
        missing.append("CUTTING_SPEED_MM_PER_MIN")
    for method, value in EJECTION_TIME_S.items():
        if value is None:
            missing.append(f'EJECTION_TIME_S["{method}"]')
    if SHEET_CHANGE_TIME_S is None:
        missing.append("SHEET_CHANGE_TIME_S")
    if missing:
        raise ValueError("Bitte in Zelle 'Maschinendaten' eintragen: " + ", ".join(missing))

## 2. Read data

**What happens:** The sheet `Daten` of the Excel file is read. The numeric columns are converted to real numbers. A decimal comma (e.g. `305,1401348`) becomes a decimal point if the values are stored as text. Anything unreadable becomes `NaN` and is reported in step 3.

**Builds on:** `customers_sales_with_dxf_data.xlsx` (output of the previous notebook) with the columns `customer_name`, `customer_branch`, `file_name`, `order_quantity`, `bounding_box_width`, `bounding_box_height`, `bounding_box_area`, `cutting_length`, `num_holes`

**Delivers:** `df` with one row per order (nothing is deleted)

**Note:** If numbers with a thousands dot were stored as text (`"1.600,5"`), they become `NaN` here instead of being read wrongly, and are reported in step 3.

In [2]:
import pandas as pd
import numpy as np

INPUT_FILE = "customers_sales_with_dxf_data.xlsx"

df = pd.read_excel(INPUT_FILE, sheet_name="Daten")
df.columns = df.columns.str.strip()

required = ["customer_name", "customer_branch", "file_name", "order_quantity",
            "bounding_box_width", "bounding_box_height", "bounding_box_area",
            "cutting_length"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Fehlende Spalten in der Excel: {missing_cols}"


def to_num(series):
    """Zahlen oder Text mit Dezimalkomma -> float (nicht lesbar -> NaN)."""
    if series.dtype == object or str(series.dtype).startswith("string"):
        series = series.astype("string").str.strip().str.replace(",", ".", regex=False)
    return pd.to_numeric(series, errors="coerce")


num_cols = ["order_quantity", "bounding_box_width", "bounding_box_height",
            "bounding_box_area", "cutting_length"]
for c in num_cols:
    df[c] = to_num(df[c])

print(df.shape)
df.head()

(75, 12)


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,num_holes,rectangularity
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,488291.056038,3438.024977,0,0.082199
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,284668.117362,3217.207505,5,0.724271
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,23799.680000,1742.517642,7,0.600032
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,26830.297074,1302.774283,2,0.685885
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,69355.398059,2764.783512,11,0.865878


## 3. Validate input data

**What happens:** Every order is checked for values required by the calculation. Orders with problems are **not deleted**; they get a reason in `status` and receive no time (`NaN`).

| Check | Meaning |
|---|---|
| missing values | a required column is empty or unreadable |
| quantity | `order_quantity` ≤ 0 or not a whole number |
| geometry | width, height, area or cutting length ≤ 0 |
| area consistency | hint only: `bounding_box_area` differs by more than 1 % from width × height |

**Builds on:** `df` from step 2

**Delivers:** `calc` (working copy of `df`) with the column `status` (empty = fine so far)

In [3]:
calc = df.copy()
calc["status"] = ""

needed = ["order_quantity", "bounding_box_width", "bounding_box_height",
          "bounding_box_area", "cutting_length"]

# 1) fehlende Werte
missing_mask = calc[needed].isna().any(axis=1)
calc.loc[missing_mask, "status"] = "Eingabewert fehlt/nicht lesbar"

# 2) ungültige Menge
bad_qty = ~missing_mask & ((calc["order_quantity"] <= 0) | (calc["order_quantity"] % 1 != 0))
calc.loc[bad_qty, "status"] = "Ungültige Menge"

# 3) ungültige Geometrie
geo = ["bounding_box_width", "bounding_box_height", "bounding_box_area", "cutting_length"]
bad_geo = ~missing_mask & ~bad_qty & (calc[geo] <= 0).any(axis=1)
calc.loc[bad_geo, "status"] = "Ungültige Geometrie (Wert <= 0)"

# 4) Hinweis: Fläche passt nicht zu Breite x Höhe (nur Warnung, Zeile bleibt im Rennen)
area_check = calc["bounding_box_width"] * calc["bounding_box_height"]
area_diff = ((calc["bounding_box_area"] - area_check).abs() / area_check) > 0.01

print(f"Orders gesamt:                     {len(calc)}")
print(f"  fehlende/nicht lesbare Werte:    {int(missing_mask.sum())}")
print(f"  ungültige Menge:                 {int(bad_qty.sum())}")
print(f"  ungültige Geometrie:             {int(bad_geo.sum())}")
print(f"  Hinweis Fläche != Breite*Höhe:   {int(area_diff.sum())}")

calc[calc["status"] != ""]

Orders gesamt:                     75
  fehlende/nicht lesbare Werte:    0
  ungültige Menge:                 0
  ungültige Geometrie:             0
  Hinweis Fläche != Breite*Höhe:   0


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,num_holes,rectangularity,status


## 4. Ejection method and parts per sheet

### 4a. Ejection method

**What happens:** The bounding box determines which ejection method the machine uses for each part. The part may be rotated (short and long side are compared).

| Condition | Method |
|---|---|
| fits into 160 × 160 mm **and** ≥ 30 × 30 mm | `SmartGate` |
| fits into 2000 × 1500 mm **and** ≥ 90 × 60 mm | `SortMaster Speed` |
| both apply | `OVERLAP_METHOD` |
| neither applies | no method, `status` = "Teilegröße außerhalb der Auswurf-Bereiche" (part size outside the ejection ranges) |

### 4b. Parts per sheet and required sheets

**What happens:**
- If the part does not fit on the sheet (3000 × 1500 mm, rotation allowed), `parts_per_sheet = 0` and `status` = "Teil passt nicht aufs Blech" (part does not fit on the sheet).
- `NESTING_METHOD = "area"`: `parts_per_sheet = sheet area // bounding_box_area` (integer, rounded down), as requested.
- `NESTING_METHOD = "grid"`: rectangles are packed in rows, both orientations are tested, the maximum counts.
- `sheets_needed = ceil(order_quantity / parts_per_sheet)`

**Important regarding accuracy:** The area method is an **upper bound**. For long, narrow parts it can yield considerably more parts per sheet than geometrically possible (example: part 1600 × 305 mm → area gives 9 parts, only 4 fit in rows). Therefore `parts_per_sheet_grid` is also calculated, and the number of orders where the chosen method gives more is reported. With more parts per sheet than possible, sheet changes are underestimated.

**Builds on:** `calc` from step 3, constants from step 1

**Delivers:** new columns `ejection_method`, `parts_per_sheet`, `parts_per_sheet_grid`, `sheets_needed`

In [4]:
def _short_long(w, h):
    return (min(w, h), max(w, h))


def _fits_max(w, h, max_dims):
    s, l = _short_long(w, h)
    ms, ml = _short_long(*max_dims)
    return s <= ms and l <= ml


def _fits_min(w, h, min_dims):
    s, l = _short_long(w, h)
    ns, nl = _short_long(*min_dims)
    return s >= ns and l >= nl


def get_ejection_method(w, h):
    """SmartGate / SortMaster Speed nach Teilegröße (Drehen erlaubt), sonst None."""
    smartgate = _fits_max(w, h, SMARTGATE_MAX_MM) and _fits_min(w, h, SMARTGATE_MIN_MM)
    sortmaster = _fits_max(w, h, SORTMASTER_MAX_MM) and _fits_min(w, h, SORTMASTER_MIN_MM)
    if smartgate and sortmaster:
        return OVERLAP_METHOD
    if smartgate:
        return "SmartGate"
    if sortmaster:
        return "SortMaster Speed"
    return None


def parts_per_sheet_area(w, h, area):
    """Teile pro Blech über die Flächen (ganze Zahl). 0 = passt nicht aufs Blech."""
    if not _fits_max(w, h, (SHEET_X_MM, SHEET_Y_MM)):
        return 0
    return int((SHEET_X_MM * SHEET_Y_MM) // area)


def parts_per_sheet_grid(w, h):
    """Rechtecke in Reihen packen, beide Ausrichtungen testen. 0 = passt nicht."""
    a = int(SHEET_X_MM // w) * int(SHEET_Y_MM // h)
    b = int(SHEET_X_MM // h) * int(SHEET_Y_MM // w)
    return max(a, b)


valid = calc["status"] == ""

# 4a) Auswurfmethode
calc["ejection_method"] = None
calc.loc[valid, "ejection_method"] = calc[valid].apply(
    lambda r: get_ejection_method(r["bounding_box_width"], r["bounding_box_height"]), axis=1)
calc["ejection_method"] = calc["ejection_method"].astype("string")   # None/NaN vereinheitlichen
no_method = valid & calc["ejection_method"].isna()
calc.loc[no_method, "status"] = "Teilegröße außerhalb der Auswurf-Bereiche"

# 4b) Teile pro Blech
valid = calc["status"] == ""
calc["parts_per_sheet"] = pd.NA
calc["parts_per_sheet_grid"] = pd.NA
calc.loc[valid, "parts_per_sheet_grid"] = calc[valid].apply(
    lambda r: parts_per_sheet_grid(r["bounding_box_width"], r["bounding_box_height"]), axis=1)
if NESTING_METHOD == "area":
    calc.loc[valid, "parts_per_sheet"] = calc[valid].apply(
        lambda r: parts_per_sheet_area(r["bounding_box_width"], r["bounding_box_height"],
                                       r["bounding_box_area"]), axis=1)
elif NESTING_METHOD == "grid":
    calc.loc[valid, "parts_per_sheet"] = calc.loc[valid, "parts_per_sheet_grid"]
else:
    raise ValueError('NESTING_METHOD muss "area" oder "grid" sein')

calc["parts_per_sheet"] = calc["parts_per_sheet"].astype("Int64")
calc["parts_per_sheet_grid"] = calc["parts_per_sheet_grid"].astype("Int64")

too_big = valid & (calc["parts_per_sheet"] == 0)
calc.loc[too_big, "status"] = "Teil passt nicht aufs Blech"

# benötigte Bleche
valid = calc["status"] == ""
calc["sheets_needed"] = pd.NA
calc.loc[valid, "sheets_needed"] = np.ceil(
    calc.loc[valid, "order_quantity"] / calc.loc[valid, "parts_per_sheet"].astype(float)).astype(int)
calc["sheets_needed"] = calc["sheets_needed"].astype("Int64")

# Hinweis: Flächenmethode optimistischer als geometrisch möglich?
optimistic = valid & (calc["parts_per_sheet"] > calc["parts_per_sheet_grid"])
print("Auswurfmethoden:\n", calc["ejection_method"].value_counts(dropna=False).to_string(), "\n")
print(f"Orders ohne Auswurfmethode:         {int(no_method.sum())}")
print(f"Orders, Teil passt nicht aufs Blech: {int(too_big.sum())}")
print(f"Orders, bei denen die gewählte Methode mehr Teile/Blech ergibt als die Reihen-Packung: "
      f"{int(optimistic.sum())} von {int(valid.sum())}")
calc[["file_name", "bounding_box_width", "bounding_box_height", "ejection_method",
      "parts_per_sheet", "parts_per_sheet_grid", "sheets_needed", "status"]].head(10)

Auswurfmethoden:
 ejection_method
SortMaster Speed    54
<NA>                11
SmartGate           10 

Orders ohne Auswurfmethode:         11
Orders, Teil passt nicht aufs Blech: 0
Orders, bei denen die gewählte Methode mehr Teile/Blech ergibt als die Reihen-Packung: 58 von 64


,file_name,bounding_box_width,bounding_box_height,ejection_method,parts_per_sheet,parts_per_sheet_grid,sheets_needed,status
0,Geo (1).DXF,1600.219048,305.140135,SortMaster Speed,9,4,178,
1,Geo (10).DXF,853.830000,333.401400,SortMaster Speed,15,12,644,
2,Geo (11).DXF,164.000000,145.120000,SortMaster Speed,189,180,42,
3,Geo (15).DXF,193.800891,138.442589,SortMaster Speed,167,150,24,
4,Geo (12).DXF,937.150697,74.006665,SortMaster Speed,64,60,138,
5,Geo (13).DXF,166.136100,67.260858,SortMaster Speed,402,396,20,
6,Geo (14).DXF,1591.023236,88.903140,SortMaster Speed,31,16,234,
7,Geo (17).DXF,1179.078462,305.093782,SortMaster Speed,12,9,159,
8,Geo (19).DXF,1152.727829,314.994358,SortMaster Speed,12,9,121,
9,Geo (2).DXF,1273.000000,37.643216,<NA>,<NA>,<NA>,<NA>,Teilegröße außerhalb der Auswurf-Bereiche


## 5. Time per part and total time per order

**What happens:** First `require_machine_data()` checks that all machine values from step 1 are filled in. If something is missing, the cell stops with a list of the missing values.

Then, only for orders with empty `status`:

```
cutting_time_s    = cutting_length / (CUTTING_SPEED_MM_PER_MIN / 60)
ejection_time_s   = EJECTION_TIME_S[ejection_method]
time_per_part_s   = cutting_time_s + ejection_time_s
sheet_changes     = sheets_needed            (COUNT_FIRST_SHEET_LOAD = True)
                  = sheets_needed - 1        (COUNT_FIRST_SHEET_LOAD = False)
production_time_s = order_quantity * time_per_part_s
sheet_change_s    = sheet_changes * SHEET_CHANGE_TIME_S
total_time_s      = production_time_s + sheet_change_s
```

**Builds on:** `calc` from step 4, constants from step 1

**Delivers:** `calc` with the columns `cutting_time_s`, `ejection_time_s`, `time_per_part_s`, `sheet_changes`, `production_time_s`, `sheet_change_s`, `total_time_s`, `total_time_min`, `total_time_h`. Orders with problems are kept with an empty time and their `status`.

**Note:** Setup times, start-up times, material changes and positioning times are not part of this model unless they are included in the constants you enter.

In [5]:
require_machine_data()

valid = calc["status"] == ""

calc["cutting_time_s"] = np.nan
calc["ejection_time_s"] = np.nan
calc["time_per_part_s"] = np.nan
calc["sheet_changes"] = pd.NA
calc["production_time_s"] = np.nan
calc["sheet_change_s"] = np.nan
calc["total_time_s"] = np.nan

calc.loc[valid, "cutting_time_s"] = calc.loc[valid, "cutting_length"] / (CUTTING_SPEED_MM_PER_MIN / 60)
calc.loc[valid, "ejection_time_s"] = calc.loc[valid, "ejection_method"].map(EJECTION_TIME_S).astype(float)
calc.loc[valid, "time_per_part_s"] = calc.loc[valid, "cutting_time_s"] + calc.loc[valid, "ejection_time_s"]

changes = calc.loc[valid, "sheets_needed"].astype(int)
if not COUNT_FIRST_SHEET_LOAD:
    changes = changes - 1
calc.loc[valid, "sheet_changes"] = changes
calc["sheet_changes"] = calc["sheet_changes"].astype("Int64")

calc.loc[valid, "production_time_s"] = calc.loc[valid, "order_quantity"] * calc.loc[valid, "time_per_part_s"]
calc.loc[valid, "sheet_change_s"] = changes * SHEET_CHANGE_TIME_S
calc.loc[valid, "total_time_s"] = calc.loc[valid, "production_time_s"] + calc.loc[valid, "sheet_change_s"]

calc["total_time_min"] = calc["total_time_s"] / 60
calc["total_time_h"] = calc["total_time_s"] / 3600

print(f"Berechnet: {int(valid.sum())} von {len(calc)} Orders")
calc.head()

Berechnet: 64 von 75 Orders


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,...,sheets_needed,cutting_time_s,ejection_time_s,time_per_part_s,sheet_changes,production_time_s,sheet_change_s,total_time_s,total_time_min,total_time_h
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,488291.056038,3438.024977,...,178,1.473439,8.0,9.473439,178,15157.502841,5340.0,20497.502841,341.625047,5.693751
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,284668.117362,3217.207505,...,644,1.378803,8.0,9.378803,644,90505.451037,19320.0,109825.451037,1830.424184,30.507070
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,23799.680000,1742.517642,...,42,0.746793,8.0,8.746793,42,68224.987547,1260.0,69484.987547,1158.083126,19.301385
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,26830.297074,1302.774283,...,24,0.558332,8.0,8.558332,24,34233.327342,720.0,34953.327342,582.555456,9.709258
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,69355.398059,2764.783512,...,138,1.184907,8.0,9.184907,138,80827.183530,4140.0,84967.183530,1416.119726,23.601995


## 6. Check and export

**What happens:**
1. **Safety check:** The `assert` aborts if the row count differs from `df`. No order is lost.
2. **Consistency check:** For all calculated orders, `total_time_s` must be ≥ `production_time_s`, and `sheets_needed * parts_per_sheet` must be at least `order_quantity`.
3. **Summary:** number of calculated orders, total time and time per customer.
4. **Export** to `orders_with_times.xlsx` with three sheets.

| Sheet | Content |
|---|---|
| `Daten` | all orders with all calculation columns |
| `Nicht berechnet` | orders without a time, with the reason in `status` |
| `Zeit je Kunde` | sum of the total time per customer (calculated orders only) |

**Builds on:** `calc` and `df` from the previous steps

**Delivers:** the file `orders_with_times.xlsx`

In [7]:
assert len(calc) == len(df), "Zeilenzahl hat sich verändert!"

ok = calc["status"] == ""
assert (calc.loc[ok, "total_time_s"] >= calc.loc[ok, "production_time_s"]).all()
assert (calc.loc[ok, "sheets_needed"] * calc.loc[ok, "parts_per_sheet"]
        >= calc.loc[ok, "order_quantity"]).all(), "Zu wenige Bleche berechnet!"

not_calculated = calc[~ok]
by_customer = (calc[ok].groupby("customer_name")
               .agg(orders=("file_name", "count"),
                    gesamt_stunden=("total_time_h", "sum"),
                    bleche=("sheets_needed", "sum"))
               .sort_values("gesamt_stunden", ascending=False)
               .reset_index())

print(f"Berechnet: {int(ok.sum())} | Nicht berechnet: {len(not_calculated)}")
print(f"Gesamtzeit aller berechneten Orders: {calc.loc[ok, 'total_time_h'].sum():,.1f} h")
print(by_customer.to_string(index=False))

with pd.ExcelWriter("orders_with_times.xlsx", engine="openpyxl") as writer:
    calc.to_excel(writer, sheet_name="Daten", index=False)
    not_calculated.to_excel(writer, sheet_name="Nicht berechnet", index=False)
    by_customer.to_excel(writer, sheet_name="Zeit je Kunde", index=False)

    for ws in writer.sheets.values():
        for column_cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in column_cells)
            ws.column_dimensions[column_cells[0].column_letter].width = min(width + 2, 60)

print("Export fertig: orders_with_times.xlsx")
calc

Berechnet: 64 | Nicht berechnet: 11
Gesamtzeit aller berechneten Orders: 1,239.9 h
     customer_name  orders  gesamt_stunden  bleche
      AutoParts AG      26      656.768434    9912
TechFab Industries      10      316.355760    1635
 PrecisionCut GmbH      15      171.722959     601
      MetalForm SA      13       95.022323    4719
Export fertig: orders_with_times.xlsx


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,...,sheets_needed,cutting_time_s,ejection_time_s,time_per_part_s,sheet_changes,production_time_s,sheet_change_s,total_time_s,total_time_min,total_time_h
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,4.882911e+05,3438.024977,...,178,1.473439,8.0,9.473439,178,15157.502841,5340.0,20497.502841,341.625047,5.693751
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,2.846681e+05,3217.207505,...,644,1.378803,8.0,9.378803,644,90505.451037,19320.0,109825.451037,1830.424184,30.507070
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,2.379968e+04,1742.517642,...,42,0.746793,8.0,8.746793,42,68224.987547,1260.0,69484.987547,1158.083126,19.301385
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,2.683030e+04,1302.774283,...,24,0.558332,8.0,8.558332,24,34233.327342,720.0,34953.327342,582.555456,9.709258
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,6.935540e+04,2764.783512,...,138,1.184907,8.0,9.184907,138,80827.183530,4140.0,84967.183530,1416.119726,23.601995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,AutoParts AG,Munich Branch,Geo (78).DXF,8150,AutoParts AG,Munich Branch,386.361049,290.000000,1.120447e+05,5581.874650,...,204,2.392232,8.0,10.392232,204,84696.690741,6120.0,90816.690741,1513.611512,25.226859
71,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,PrecisionCut GmbH,Stuttgart Branch,36.553638,36.553638,1.336168e+03,147.032435,...,2,0.063014,5.0,5.063014,2,18986.302128,60.0,19046.302128,317.438369,5.290639
72,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,TechFab Industries,Hamburg Branch,801.705747,344.180571,2.759315e+05,6111.748725,...,754,2.619321,8.0,10.619321,754,127962.816630,22620.0,150582.816630,2509.713610,41.828560
73,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,MetalForm SA,Frankfurt Branch,1968.901094,832.434782,1.638982e+06,16821.059941,...,1050,7.209026,8.0,15.209026,1050,31938.953947,31500.0,63438.953947,1057.315899,17.621932


In [13]:


SHEET_SIZE = (SHEET_X_MM, SHEET_Y_MM)   # <<< NEW: TruLaser Center 7030, X x Y in mm

def get_dxf_extent(path):                                           # <<< NEW
    """
    Same logic as get_dxf_bounds(), but returns the position too:
    (min_x, min_y, max_x, max_y), or None on error / empty file.
    Needed to move the part to the origin before placing it on the sheet.
    The width/height are identical to get_dxf_bounds(), so the sheet view
    matches the numbers in the order table (bounding_box_width/height).
    """
    try:
        msp = ezdxf.readfile(path).modelspace()
        xs_all, ys_all = [], []
        for e in msp:
            t = e.dxftype()
            pts = []
            if t == "LINE":
                pts = [e.dxf.start, e.dxf.end]
            elif t == "CIRCLE":
                c, r = e.dxf.center, e.dxf.radius
                pts = [(c[0] - r, c[1] - r), (c[0] + r, c[1] + r)]
            elif t == "ARC":
                pts = flatten(e)
            elif t == "LWPOLYLINE":
                pts = [(float(p[0]), float(p[1])) for p in e.get_points() if len(p) >= 2]
            elif t == "POLYLINE":
                pts = [(float(v.dxf.x), float(v.dxf.y)) for v in e.vertices if hasattr(v, "dxf")]
            for p in pts:
                xs_all.append(p[0])
                ys_all.append(p[1])
        if not xs_all:
            return None
        return min(xs_all), min(ys_all), max(xs_all), max(ys_all)
    except Exception as ex:
        print(f"Error reading {path}: {ex}")
        return None


def get_sheet_layout(width, height, sheet_size=SHEET_SIZE, gap=0.0, margin=0.0):   # <<< NEW
    """
    Places identical rectangles in rows/columns on the sheet. Both orientations are tried
    (part as drawn / rotated by 90 degrees), the one with more parts wins (tie = as drawn).
    This is the same 'grid' logic as in the order-time notebook (gap = margin = 0 there).

    Args:
        width, height: bounding box of one part (mm)
        sheet_size: (X, Y) of the sheet (mm)
        gap: free space between two parts (mm)
        margin: free border around the sheet (mm)

    Returns:
        dict with rotated, cols, rows, count, cell (w, h) and positions [(x, y), ...]
        (lower left corner of each cell, filled row by row), or None if the part does not fit.
    """
    sx, sy = sheet_size
    best = None
    for rotated in (False, True):
        pw, ph = (height, width) if rotated else (width, height)
        cols = max(int((sx - 2 * margin + gap + 1e-9) // (pw + gap)), 0)
        rows = max(int((sy - 2 * margin + gap + 1e-9) // (ph + gap)), 0)
        count = cols * rows
        if best is None or count > best["count"]:
            best = {"rotated": rotated, "cols": cols, "rows": rows, "count": count, "cell": (pw, ph)}
    if best["count"] == 0:
        return None
    pw, ph = best["cell"]
    best["positions"] = [(margin + c * (pw + gap), margin + r * (ph + gap))
                         for r in range(best["rows"]) for c in range(best["cols"])]
    return best


def _face_path_data(poly, min_x, min_y):                            # <<< NEW
    """Shapely polygon (with holes) -> matplotlib vertices/codes, moved to the part origin."""
    poly = orient(poly, 1.0)    # outer ring counter-clockwise, holes clockwise -> holes are cut out
    verts, codes = [], []
    for ring in [poly.exterior, *poly.interiors]:
        xy = np.asarray(ring.coords)[:, :2] - (min_x, min_y)
        verts.append(xy)
        codes += [Path.MOVETO] + [Path.LINETO] * (len(xy) - 2) + [Path.CLOSEPOLY]
    return np.vstack(verts), np.array(codes)


def plot_sheet(file_path, quantity=None, sheet_number=1, sheet_size=SHEET_SIZE,
               gap=0.0, margin=0.0, show_cells=True, save_path=None):            # <<< NEW
    """
    Shows a sheet with the placed copies of one part, to see how production would run.

    Args:
        file_path: DXF file of the part.
        quantity: order quantity. With it, the number of sheets is calculated and the
                  selected sheet shows only the parts that are still needed (last sheet is
                  usually not full). Without it, one full sheet is shown.
        sheet_number: which sheet to show (1 = first; -1 = last).
        sheet_size: (X, Y) of the sheet in mm.
        gap / margin: free space between parts / at the sheet border (mm). Default 0,
                      like in the time calculation.
        show_cells: draw the bounding box of every placed part (dashed).
        save_path: optional, saves the picture (e.g. "sheet.png").

    Returns:
        dict with part size, rotation, parts_per_sheet, sheets_needed, parts_on_sheet, utilization,
        area_method_count (what the area formula would assume) - or None if nothing can be shown.

    Note: Placement is a simple grid of identical rectangles (not nested, as agreed).
    Only closed contours are drawn; open lines (e.g. engraving) are not shown.
    """
    extent = get_dxf_extent(file_path)
    if extent is None:
        print(f"No geometry found in {file_path}")
        return None
    min_x, min_y, max_x, max_y = extent
    w, h = max_x - min_x, max_y - min_y
    if w <= 0 or h <= 0:
        print(f"Invalid bounding box in {file_path}: {w} x {h}")
        return None

    layout = get_sheet_layout(w, h, sheet_size, gap, margin)
    if layout is None:
        print(f"{os.path.basename(file_path)}: part {w:.1f} x {h:.1f} mm does not fit on the "
              f"{sheet_size[0]} x {sheet_size[1]} mm sheet.")
        return None
    per_sheet = layout["count"]

    # --- how many sheets, how many parts on the selected sheet ---
    if quantity is None:
        sheets_needed, on_sheet, sheet_number = 1, per_sheet, 1
    else:
        quantity = int(quantity)
        if quantity <= 0:
            raise ValueError("quantity must be > 0")
        sheets_needed = math.ceil(quantity / per_sheet)
        if sheet_number < 0:                                        # -1 = last sheet
            sheet_number = sheets_needed + 1 + sheet_number
        if not 1 <= sheet_number <= sheets_needed:
            raise ValueError(f"sheet_number must be between 1 and {sheets_needed}")
        on_sheet = per_sheet if sheet_number < sheets_needed else quantity - (sheets_needed - 1) * per_sheet

    # --- geometry of one part (material faces incl. their holes), moved to the origin ---
    faces = [f for f, depth in get_faces(file_path) if depth % 2 == 0]
    if not faces:
        print(f"No closed contour found in {file_path}")
        return None
    base = [_face_path_data(f, min_x, min_y) for f in faces]
    material_area = sum(f.area for f in faces)

    patches, cells = [], []
    pw, ph = layout["cell"]
    for ox, oy in layout["positions"][:on_sheet]:
        for verts, codes in base:
            v = np.column_stack((h - verts[:, 1], verts[:, 0])) if layout["rotated"] else verts
            patches.append(PathPatch(Path(v + (ox, oy), codes)))
        cells.append(Rectangle((ox, oy), pw, ph))

    # --- plot ---
    sx, sy = sheet_size
    fig, ax = plt.subplots(figsize=(12, 12 * sy / sx + 1.2))
    ax.add_patch(Rectangle((0, 0), sx, sy, facecolor="#f4f4f4", edgecolor="black", linewidth=1.5))
    if show_cells:
        ax.add_collection(PatchCollection(cells, facecolor="none", edgecolor="#999999",
                                          linewidth=0.4, linestyle="--"))
    ax.add_collection(PatchCollection(patches, facecolor="#9ecae1", edgecolor="black", linewidth=0.5))
    pad = 0.02 * sx
    ax.set_xlim(-pad, sx + pad)
    ax.set_ylim(-pad, sy + pad)
    ax.set_aspect("equal", "box")
    ax.set_xlabel("X [mm]")
    ax.set_ylabel("Y [mm]")

    utilization = on_sheet * material_area / (sx * sy)
    area_count = int((sx * sy) // (w * h))
    title = (f"{os.path.basename(file_path)} - sheet {sheet_number}/{sheets_needed} "
             f"({sx} x {sy} mm)\n"
             f"{on_sheet} of {per_sheet} places used, "
             f"{'rotated by 90 degrees' if layout['rotated'] else 'not rotated'}, "
             f"part {w:.1f} x {h:.1f} mm, material utilisation {utilization:.0%}")
    if area_count > per_sheet:
        title += (f"\nNote: the area formula would assume {area_count} parts per sheet, "
                  f"geometrically only {per_sheet} fit")
    ax.set_title(title, fontsize=10)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    return {
        "file": os.path.basename(file_path),
        "part_size": (w, h),
        "rotated": layout["rotated"],
        "parts_per_sheet": per_sheet,
        "sheets_needed": sheets_needed,
        "sheet_number": sheet_number,
        "parts_on_sheet": on_sheet,
        "utilization": utilization,
        "area_method_count": area_count,
    }


def _natural_key(text):                                             # <<< NEW
    """Sort 'Geo (2).DXF' before 'Geo (10).DXF'."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(text))]


def choose_and_plot_sheet(folder, orders=None, **plot_kwargs):      # <<< NEW
    """
    The user picks a file (or an order) and the sheet with the placed parts is shown.

    Args:
        folder: folder with the DXF files, e.g. "10991360".
        orders: optional DataFrame (e.g. `merged`) with the columns file_name and order_quantity
                (customer_name / customer_branch are shown if present). Then the user picks an
                ORDER, so the number of sheets comes from the order quantity.
                Without it the user picks a DXF file and one full sheet is shown.
        **plot_kwargs: passed to plot_sheet (sheet_size, gap, margin, show_cells, save_path).

    In Jupyter a dropdown with a "Show sheet" button is shown (needs ipywidgets, which comes
    with `jupyter`). Without ipywidgets a numbered text menu is used.
    """
    entries = []                                                    # (label, path, quantity)
    if orders is None:
        for f in sorted(os.listdir(folder), key=_natural_key):
            if f.lower().endswith(".dxf"):
                entries.append((f, os.path.join(folder, f), None))
    else:
        skipped = 0
        for _, row in orders.iterrows():
            path = os.path.join(folder, str(row["file_name"]))
            if not os.path.isfile(path):
                skipped += 1
                continue
            qty = row["order_quantity"]
            qty = None if qty != qty else int(qty)                  # NaN -> None
            parts = [str(row[c]) for c in ("customer_name", "customer_branch") if c in row.index]
            parts += [str(row["file_name"]), f"qty {qty}"]
            entries.append((" | ".join(parts), path, qty))
        if skipped:
            print(f"{skipped} orders skipped: DXF file not found in '{folder}'")
    if not entries:
        print("Nothing to choose from.")
        return None

    def show(index, sheet_number):
        _, path, qty = entries[index]
        try:
            plot_sheet(path, quantity=qty, sheet_number=sheet_number, **plot_kwargs)
        except ValueError as ex:
            print(ex)

    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
    except ImportError:                                             # text menu fallback
        for i, (label, _, _) in enumerate(entries):
            print(f"{i + 1:4d}  {label}")
        index = int(input("Number: ")) - 1
        sheet_number = int(input("Sheet number [1]: ") or 1)
        show(index, sheet_number)
        return None

    dropdown = widgets.Dropdown(options=[(label, i) for i, (label, _, _) in enumerate(entries)],
                                description="Order:" if orders is not None else "File:",
                                layout=widgets.Layout(width="95%"))
    sheet_input = widgets.BoundedIntText(value=1, min=-10**6, max=10**6, description="Sheet no.:")
    button = widgets.Button(description="Show sheet", button_style="primary")
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output(wait=True)
            show(dropdown.value, sheet_input.value)

    button.on_click(on_click)
    display(widgets.VBox([dropdown, widgets.HBox([sheet_input, button]), out]))
    return None

In [22]:
SHEET_SIZE = (SHEET_X_MM, SHEET_Y_MM)   # <<< NEW: TruLaser Center 7030, X x Y in mm
orders_xlsx = pd.read_excel("customers_sales_with_dxf_data.xlsx", sheet_name="Daten")
choose_and_plot_sheet("10991360", orders=orders_xlsx)
plot_sheet("10991360/Geo (10).DXF", SHEET_SIZE, quantity=9650, sheet_number=-1)

TypeError: plot_sheet() got multiple values for argument 'quantity'